# AI-powered 6G RAN Optimization - Colab Notebook

This notebook generates synthetic RAN data, trains models, runs inference, and visualizes outcomes.

**Run the setup cell first** - it clones the repository (when needed) and makes the project packages importable.

In [ ]:
# Setup: make the project packages (data, pipeline, ...) importable.
# Works both in Google Colab (clones the repo) and locally (uses the repo you already have).
import os
import sys

REPO_URL = "https://github.com/erdioz/AI-powered-6G-RAN-Optimization-System.git"
REPO_DIR = "AI-powered-6G-RAN-Optimization-System"

# If we are not already inside the project root, fetch/enter it.
if not (os.path.isdir("data") and os.path.isfile("pyproject.toml")):
    if not os.path.isdir(REPO_DIR):
        !git clone -q {REPO_URL}
    os.chdir(REPO_DIR)

# Ensure the project root is on sys.path so `import data`, `import pipeline`, ... work.
project_root = os.getcwd()
if project_root not in sys.path:
    sys.path.insert(0, project_root)

print("Working directory:", project_root)

In [ ]:
# Install dependencies. Editable install also registers the `ran6g` CLI.
!pip -q install -e . || pip -q install -r requirements.txt

In [ ]:
from data.generator import SyntheticRANDataGenerator
from pipeline.trainer import train_all
from pipeline.inference import RANInferenceService
from visualization.plots import plot_ue_movement, plot_sinr_over_time, plot_beam_selection, plot_anomalies
import pandas as pd
from IPython.display import Image, display

In [ ]:
generator = SyntheticRANDataGenerator()
df = generator.generate()
df.to_csv('data/sample_dataset.csv', index=False)
df.head()

In [ ]:
metrics = train_all('data/sample_dataset.csv')
metrics

In [ ]:
service = RANInferenceService()
sample = df.iloc[0].to_dict()

# Build each payload from the model's own feature list so it stays in sync with the models.
qos_payload = {k: sample[k] for k in service.qos.config.feature_columns}
beam_payload = {k: sample[k] for k in service.beam.config.feature_columns}
anom_payload = {k: sample[k] for k in service.anomaly.config.feature_columns}

print(service.predict_qos(qos_payload))
print(service.select_beam(beam_payload))
print(service.detect_anomaly(anom_payload))

In [ ]:
plot_ue_movement(df, user_id=0)
plot_sinr_over_time(df, user_id=0)
plot_beam_selection(df, user_id=0)
plot_anomalies(df)
display(Image('outputs/plots/ue_movement.png'))
display(Image('outputs/plots/sinr_over_time.png'))
display(Image('outputs/plots/beam_selection.png'))
display(Image('outputs/plots/anomalies.png'))